In [2]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, Point
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.image as mpimg
from scipy.ndimage import zoom
import matplotlib.patches as patches

# Enhanced colormap with more detailed gradient
colors = ["#f2f2f2","#b5c9fd","#9fbefd","#889eea","#6171f7","#3e55f4","#009694","#0cff00","#e6ff00","#ffff00","#ffcf00","#ff9f00","#ff6f00","#ff1b00","#e60000","#cc0000","#a600a4","#c27ec1","#e094c3","#ffbffd","#f5a6f9","#6fc3fb","#0098fe"]
cmap_custom = mcolors.LinearSegmentedColormap.from_list('custom_cmap', colors, N=300)

temperature_colors = [
    '#8A2BE2', '#6A5ACD', '#483D8B',  # Violet to Dark Blue (7 colors)
    '#ADD8E6', '#B0E0E6', '#00BFFF',  # Light Blue (6 colors)
    '#00CED1', '#20B2AA', '#008B8B',  # Greenish Blue (6 colors)
    '#3CB371', '#32CD32', '#228B22',  # Green (6 colors)
    '#FFFF00', '#FFD700',             # Yellow (2 colors)
    '#FFA500', '#FF8C00', '#FF4500',  # Orange (6 colors)
    '#8B0000', '#800000', '#A52A2A'   # Dark Red to Red (6 colors)
]

wind_max_2m_colors = [
    '#FFFFFF',                          # White (0 km/h)
    '#00FF00', '#00E600', '#00CC00',   # Green (0-50 km/h)
    '#FFFF00', '#FFCC00',              # Yellow (50-100 km/h)
    '#FFA500', '#FF8C00',              # Orange (100-150 km/h)
    '#FF0000', '#CC0000',              # Red (150-200 km/h)
    '#FFFF00'                          # Yellow (>200 km/h)
]

cmap_temperature = mcolors.LinearSegmentedColormap.from_list('temperature_cmap', temperature_colors, N=300)
cmap_wind_max_2m_colors = mcolors.LinearSegmentedColormap.from_list('wind_max_2m_cmap', wind_max_2m_colors, N=300)

# Define custom tick labels for precipitation
precip_ticks = [0, 1, 2, 5, 10, 15, 20, 30, 40, 50, 60, 80, 100, 120, 140, 160, 180, 200, 250, 300, 350]

# Define the specific contour levels for precipitation
precip_contour_levels = [0, 1, 2, 5, 10, 15, 20, 30, 40, 50, 60, 80, 100, 120, 140, 160, 180, 200, 250, 300, 350]

# Function to create the plot
def create_variable_plot(df, variable_name, title, x_title, colormap, legend_ticks, value_range, output_filename):
    if variable_name not in df.columns:
        print(f"Variable '{variable_name}' not found in the CSV file. Skipping...")
        return
    
    # Read the variable data
    variable_values = df.pivot("Latitude", "Longitude", variable_name).values

    lat_values = df['Latitude'].unique()
    lon_values = df['Longitude'].unique()

    # Load the world map with medium resolution
    world = gpd.read_file('../data/shapes/shape/ne_10m_admin_0_countries/ne_10m_admin_0_countries.shp')

    # Define the neighboring countries
    countries = ['Slovenia', 'Austria', 'Italy', 'Hungary', 'Croatia']

    # Filter the neighboring countries
    region = world[world['NAME'].isin(countries)]

    # Create bounding box for the region
    latitude_center, longitude_center = 46.1512, 14.9955
    lat_min = latitude_center - 1
    lat_max = latitude_center + 1
    lon_min = longitude_center - 2
    lon_max = longitude_center + 1.80

    bbox_polygon = Polygon([(lon_min, lat_min), (lon_min, lat_max), (lon_max, lat_max), (lon_max, lat_min)])

    # Adjust the bounding box of the region to match the defined bounding box
    region_clipped = gpd.clip(region, bbox_polygon)

    # Plot the bounding areas with detailed geometries
    fig, ax = plt.subplots(figsize=(15, 15))
    region_clipped.plot(ax=ax, color='lightgrey', edgecolor='black')
    gpd.GeoSeries(bbox_polygon).boundary.plot(ax=ax, color='#333333', linestyle='--')
    fig.set_facecolor('#333333')

    # Clip variable values within the desired range
    if variable_name == "Total_Precipitation":
        variable_clipped = np.clip(variable_values, value_range[0], value_range[1])
        contour_levels = precip_contour_levels
    elif variable_name == "windgusts_10m_max":
        variable_clipped = np.clip(variable_values, value_range[0], value_range[1])
        contour_levels = np.arange(legend_ticks[0], legend_ticks[-1] + 1, step=10)
    else:
        variable_clipped = np.clip(variable_values, value_range[0], value_range[1])
        contour_levels = np.arange(legend_ticks[0], legend_ticks[-1] + 2, step=2)

    # Identify the maximum variable value and its coordinates
    max_value = variable_clipped.max()
    max_value_coords = np.unravel_index(variable_clipped.argmax(), variable_clipped.shape)

    # Use the specified colormap in the contour plot
    contour_plot = ax.contourf(lon_values, lat_values, variable_clipped, levels=contour_levels, cmap=colormap, extend='both')

    # Optionally, set stride to control the density of the text
    stride = 4

    # Iterate through the grid and add text for variable values
    for i in range(0, len(lat_values), stride):
        for j in range(0, len(lon_values), stride):
            # Skip values outside of the bounding box
            if not bbox_polygon.contains(Point(lon_values[j], lat_values[i])):
                continue
            # Extract variable value and convert to integer
            var_val = int(round(variable_clipped[i, j]))
            # Always plot the maximum value
            if (i, j) == max_value_coords or var_val != 0:
                ax.text(lon_values[j], lat_values[i], f'{var_val}', fontsize=8, ha='center', va='center', color='black')

    # Add borders between countries
    region_clipped.boundary.plot(ax=ax, linewidth=1, color='black')

    plt.xlim(lon_min, lon_max)
    plt.ylim(lat_min, lat_max)

    # Remove x and y axis
    plt.xticks([])
    plt.yticks([])

    # Load and resize the logo (replace 'logo.png' with your actual logo path)
    logo = mpimg.imread('../assets/logo/logo.png')

    # Scale down the logo by a factor; you can adjust this value
    scaling_factor = 1
    logo_resized = zoom(logo, (scaling_factor, scaling_factor, 1))

    # Add the logo to the figure; adjust coordinates to place at the top left
    fig.figimage(logo_resized, xo=60, yo=3070, zorder=20)

    # Set title and source information
    title_font = {'size':'19', 'color':'white', 'weight':'light'}
    fig.text(0.5, 0.795, title, ha='center', **title_font)
    fig.text(0.906, 0.12, "Vir podatkov: Open-Meteo", ha="right", fontsize=10, color="white")
    fig.text(0.12, 0.12, "Napovedni model: ICON-D2", ha="left", fontsize=10, color="white", )
    fig.text(0.899, 0.794, f'Veljavnost: 19.8.2023', ha='right', fontsize=10, color="white")

    cax_height = 0.02

    # Create axes for the colorbar to make it the same width as the plot, and place at the very bottom
    cax = fig.add_axes([0.15, 0.17, 0.7, cax_height])
    
    cbar = plt.colorbar(contour_plot, cax=cax, orientation='horizontal', ticks=legend_ticks, label=x_title)
    
    # Adjust the width of the colorbar
    cbar.ax.set_position([cax.get_position().x0 - 0.025, cax.get_position().y0 - 0, cax.get_position().width + 0.075, cax.get_position().height])
    
    # Set the colorbar tick label color
    cbar.ax.xaxis.set_tick_params(color='white')
    # Set the color of the tick labels
    cbar.set_ticklabels([str(int(level)) for level in legend_ticks], color='white')

    # Set the colorbar label color
    cbar.set_label(x_title, color='white', labelpad=10)

    # Format the tick labels
    if variable_name == "Total_Precipitation":
        cax.set_xticks(precip_contour_levels)
        cax.set_xticklabels([str(int(level)) for level in precip_contour_levels])
    else:
        step = 5
        cax.set_xticks(contour_levels)
        cax.set_xticklabels([str(int(level)) for level in contour_levels])

    # Make the border white and add padding
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')
        spine.set_linewidth(3)  # Adjust the border thickness to your liking

    # You may also want to add a padding around the plot to prevent clipping
    ax.margins(x=0.05, y=0.05)  # You can adjust the padding by changing the values

    # Set the tick color for both x and y axes
    ax.tick_params(axis='x', colors='white')
    ax.tick_params(axis='y', colors='white')

    # Save the figure
    plt.savefig(output_filename, bbox_inches='tight', dpi=300)
    plt.close()

# List of variable names, titles, colormaps, legend ticks, and value ranges
variables_and_settings = [
    ("Temperature (Celsius)", "Maksimalna temperatura zraka na višini 2 m", "Temperatura zraka [°C]", cmap_temperature, list(range(-20, 40, 1)), (-20, 40)),
    ("temperature_2m_min", "Minimalna temperatura zraka na višini 2 m", "Temperatura zraka [°C]", cmap_temperature, list(range(-20, 40, 1)), (-20, 40)),
    ("windgusts_10m_max", "Maksimalni sunki vetra na višini 10 m", "Sunki vetra [km/h]", cmap_wind_max_2m_colors, list(range(0, 200, 10)), (0, 200)),
    ("Total_Precipitation", "Napoved količine padavin", "Količina padavin [mm]",  cmap_custom, precip_ticks, (0, 350)),
]

# Loop through the variables and create plots
for variable, title, x_title, colormap, legend_ticks, value_range in variables_and_settings:
    input_filename = f'../31_grib_easy_bounding/output_data.csv'
    output_filename = f'{variable}_plot.png'
    df = pd.read_csv(input_filename)
    create_variable_plot(df, variable, title, x_title, colormap, legend_ticks, value_range, output_filename)

C:\Users\admin\AppData\Local\Temp\ipykernel_1736\4219062392.py:50: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  variable_values = df.pivot("Latitude", "Longitude", variable_name).values
c:\Users\admin\miniconda3\envs\tf\lib\site-packages\geopandas\tools\clip.py:67: FutureWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  clipped.loc[


Variable 'temperature_2m_min' not found in the CSV file. Skipping...
Variable 'windgusts_10m_max' not found in the CSV file. Skipping...
Variable 'Total_Precipitation' not found in the CSV file. Skipping...
